# CLV 이중축 M2 실효강도 정규화 진단
기존 seed 42·43·44의 주모형과 두 대조군 checkpoint를 재학습 없이 평가합니다. 모델별 residual 크기를 M1 점수 대비 동일한 `rho`로 맞추며, Dunnhumby 전체기간과 H&M 60일 validation만 순차 실행합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import importlib, os, shutil, subprocess, sys
REVIEWED_SHA = '93dbb164b98897e33adf2c294551bda668ff7abc'
REPO_DIR = '/content/clv-m2-lightgcn-runner'
os.chdir('/content')
shutil.rmtree(REPO_DIR, ignore_errors=True)
subprocess.run(['git', 'clone', '-q', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', REPO_DIR], check=True)
os.chdir(REPO_DIR)
subprocess.run(['git', 'checkout', '-q', REVIEWED_SHA], check=True)
for name in tuple(sys.modules):
    if name.startswith('lightgcn_clv') or name == 'clv_dual_axis_model':
        sys.modules.pop(name, None)
importlib.invalidate_caches()
actual_sha = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
assert actual_sha == REVIEWED_SHA
print('코드 준비 완료:', actual_sha)

In [ ]:
from pathlib import Path
from IPython.display import display
import inspect, torch
import lightgcn_clv_dual_normalized_strength as normalized

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
ROOT = Path('/content/drive/MyDrive/논문/data')
DUN = ROOT / 'results_clv_dual_dunnhumby'
HM = ROOT / 'results_clv_dual_hm_w60'
configs = {
    'dunnhumby': normalized.configure_normalized_strength(
        'dunnhumby',
        DUN / 'clv_dual_dunnhumby_662adb04aa.json',
        DUN / 'multiseed_validation/clv_dual_multiseed_dunnhumby.json',
        DUN / 'multiseed_control_validation/clv_dual_multiseed_controls_dunnhumby.json',
        out_dir=DUN / 'normalized_strength',
    ),
    'hm_w60': normalized.configure_normalized_strength(
        'hm',
        HM / 'clv_dual_hm_fdb66ee106.json',
        HM / 'multiseed_validation/clv_dual_multiseed_hm.json',
        HM / 'multiseed_control_validation/clv_dual_multiseed_controls_hm.json',
        short_hm=True,
        out_dir=HM / 'normalized_strength',
    ),
}
for name, cfg in configs.items(): print(name, cfg)

## 평가 실행
학습은 수행하지 않습니다. 기존 checkpoint를 불러와 `rho = 0.2, 0.4, 0.6, 0.8, 1.0`에서 평가만 수행합니다.

In [ ]:
normalized = importlib.reload(normalized)
load_model_signature = tuple(inspect.signature(normalized._load_model).parameters)
assert load_model_signature == ('prepared', 'run_cfg', 'gate_shape', 'seed', 'model_id', 'checkpoint'), load_model_signature
print('정규화 실행 코드 확인:', normalized.__file__, load_model_signature)
results = {}
for name in ('dunnhumby', 'hm_w60'):
    print(f'\n===== {name}: 정규화 곡선 시작 =====')
    results[name] = normalized.run_normalized_strength(configs[name])
    print(f'===== {name}: 완료 =====')

In [ ]:
for name, frame in results.items():
    print(f'\n===== {name}: rho 곡선 =====')
    display(frame[['seed', 'model_id', 'rho', 'lambda_equivalent', 'raw_effective_ratio', 'effective_strength', 'recall@10', 'ndcg@10', 'revenue@10', 'coverage@10', 'n_distinct@10']].sort_values(['rho', 'seed', 'model_id']))
    decision = frame.attrs['normalized_strength_decision']
    print('정규화 모델 채택조건 통과:', decision['success'])
    print('선택 rho:', decision['selected_rho'])
    print('실패조건:', decision['failed_conditions'])
    display(__import__('pandas').DataFrame(decision['rho_table']))
    print('결과 파일:', frame.attrs['result_paths'])
print('완료: 결과를 검토하기 전에는 H&M 2년이나 test를 실행하지 않습니다.')